# Validate linear summation (metamodel vs dynamic)

Compare offshore-driven **dynamic SnapWave** to the **reconstructed metamodel**
(spectral mode: WHACS partitions `0`–`3` per goal, wavespectra linear sum).

Bulk output is configured in the case JSON → `metamodel.reconstruction.variables`
(HyGloW name → wavespectra stat). Set `save_spectra: true` to cache summed spectra
for notebook 11.

Comparison uses `hs`, `tp`, `dir`, and `spr` when both datasets contain them
(`val.resolve_comparison_variables`).

**Prerequisites:** fitted models (06), dynamic run (08). Use `recompute=True` after refitting.

In [ ]:
import matplotlib.pyplot as plt
import time
import xarray as xr
from utils import simulation as sim
from utils import validation as val
from utils import metamodel as mm

## Configuration

In [ ]:
period = "2020-07"
offshore_name = "WHACS"
mda_n = 450  # must match a fitted mda_candidates entry
mm_set = "buoys"  # metamodel set (metamodel.sets); `sites` must belong to it

# Site subset — isobath depth, mesh polygon, buoy id, or umbrella
sites = "buoy"  # e.g. "10m", "buoy", or a list of explicit site ids

# Metrics to compare (subset used when missing from either dataset)
compare_variables = ["hs", "tp", "dir", "spr"]

dynamic_path = (
    sim.cases_dir("dynamic") / f"Dynamic_Cases_{offshore_name}_2020.nc"
)

print(f"period       : {period}")
print(f"mda_n        : {mda_n}")
print(f"metamodel    : {mm_set}")
print(f"sites        : {sites}")
print(f"compare vars : {compare_variables}")
print(f"dynamic path : {dynamic_path}")

## 1 · Check models

In [ ]:
mm.require_models(mm_set, mda_n)
print(f"sites of {mm_set!r} kept in every goal: {len(mm.reconstruction_sites(mm_set))}")

## 2 · Reconstruct metamodel (bulk)

In [ ]:
t0 = time.perf_counter()
metamodel = mm.reconstruct(
    period,
    mda_n,
    metamodel=mm_set,
    spectral=False,
    offshore_name=offshore_name,
    sites=sites,
    workers=4,
    save_spectra=False,  # bulk-vs-dynamic validation never reads spectra —
    # the case JSON defaults save_spectra=true (for notebook 11), which was
    # needlessly computing and accumulating a full (time, sites, freq, dir)
    # array per batch here, ~1000x bigger than the bulk stats actually
    # used below, and is the real driver of memory growth if this is run
    # at full site-count scale.
)
reconstruct_seconds = time.perf_counter() - t0
print(f"Reconstruction took {reconstruct_seconds:.2f}s for {metamodel.sizes.get('sites', '?')} sites")
metamodel

## 3 · Load dynamic reference

In [ ]:
dynamic = xr.open_dataset(dynamic_path).sel(time=period)
dynamic = dynamic.sel(sites=metamodel.sites.values).load()
dynamic

## 4 · Compare

In [ ]:
variables = val.resolve_comparison_variables(
    metamodel,
    dynamic,
    variables=compare_variables,
)
print(f"Comparing: {variables}")

result = val.compare(
    reference=dynamic,
    estimate=metamodel,
    variables=variables,
    reference_label=val.DYNAMIC_REFERENCE_LABEL,
    estimate_label="Linear summation",
)
result.metrics

## 5 · Save metrics

In [ ]:
metrics_out = result.metrics.copy()
metrics_out.insert(0, "period", period)
metrics_out.insert(1, "metamodel", mm_set)
metrics_out.insert(2, "mda_candidate", mda_n)
metrics_out.insert(3, "offshore_name", offshore_name)
metrics_out["n_sites"] = int(metamodel.sizes["sites"])
# Wall time for mm.reconstruct() above — near-zero if these sites/period
# were already cached from a previous run, not a fresh-simulation cost.
metrics_out["reconstruct_seconds"] = reconstruct_seconds

out_path = val.metamodel_validation_path(period, mm_set)
out_path.parent.mkdir(parents=True, exist_ok=True)
metrics_out.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

In [ ]:
result.plot_all(variables=variables)
plt.show()